# 04 - Modelo DL Treinado do Zero: CNN
A CNN (Convolutional Neural Network) é uma rede neural originally criada para visão computacional, mas que também apresenta excelente desempenho em classificação de texto. Foi treinada do zero com nosso dataset, utilizando filtros convolucionais para detectar padrões locais nas sequências de palavras dos e-mails.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import joblib

print(f"TensorFlow versão: {tf.__version__}")
print("Bibliotecas carregadas!")

TensorFlow versão: 2.21.0
Bibliotecas carregadas!


In [2]:
# Carrega o dataset limpo
df = pd.read_csv('../data/phishing_email_limpo.csv')

# Reutiliza o mesmo tokenizador do LSTM para consistência
tokenizer = joblib.load('../models/lstm_tokenizer.pkl')

VOCAB_SIZE = 10000
MAX_LEN = 200

# Transforma os textos em sequências de números
sequences = tokenizer.texts_to_sequences(df['text_combined'])
X = pad_sequences(sequences, maxlen=MAX_LEN, truncating='post', padding='post')
y = df['label'].values

# Divide em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Treino: {X_train.shape[0]} e-mails")
print(f"Teste:  {X_test.shape[0]} e-mails")

Treino: 65922 e-mails
Teste:  16481 e-mails


In [3]:
# Constrói o modelo CNN para classificação de texto
model_cnn = Sequential([
    # Embedding: transforma os números em vetores densos
    Embedding(input_dim=VOCAB_SIZE, output_dim=64),
    
    # Conv1D: aplica filtros para detectar padrões locais no texto
    Conv1D(filters=128, kernel_size=5, activation='relu'),
    
    # GlobalMaxPooling1D: pega o valor mais importante de cada filtro
    GlobalMaxPooling1D(),
    
    # Dropout: evita overfitting
    Dropout(0.5),
    
    # Camada de saída
    Dense(1, activation='sigmoid')
])

model_cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model_cnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d (Conv1D)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_max_pooling1d                 │ ?                           │               0 │
│ (GlobalMaxPooling1D)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [4]:
history_cnn = model_cnn.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

print("Treinamento concluído!")

Epoch 1/5
928/928 ━━━━━━━━━━━━━━━━━━━━ 43s 42ms/step - accuracy: 0.9523 - loss: 0.1247 - val_accuracy: 0.9830 - val_loss: 0.0481
Epoch 2/5
928/928 ━━━━━━━━━━━━━━━━━━━━ 40s 43ms/step - accuracy: 0.9880 - loss: 0.0350 - val_accuracy: 0.9847 - val_loss: 0.0429
Epoch 3/5
928/928 ━━━━━━━━━━━━━━━━━━━━ 40s 43ms/step - accuracy: 0.9937 - loss: 0.0184 - val_accuracy: 0.9844 - val_loss: 0.0428
Epoch 4/5
928/928 ━━━━━━━━━━━━━━━━━━━━ 41s 43ms/step - accuracy: 0.9967 - loss: 0.0101 - val_accuracy: 0.9870 - val_loss: 0.0470
Epoch 5/5
928/928 ━━━━━━━━━━━━━━━━━━━━ 40s 43ms/step - accuracy: 0.9980 - loss: 0.0066 - val_accuracy: 0.9842 - val_loss: 0.0646
Treinamento concluído!


In [5]:
y_pred_prob = model_cnn.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

print(f"Acurácia: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("Relatório de Classificação:")
print(classification_report(y_test, y_pred, target_names=['Legítimo', 'Phishing']))

516/516 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step
Acurácia: 98.32%

Relatório de Classificação:
              precision    recall  f1-score   support

    Legítimo       0.99      0.97      0.98      8027
    Phishing       0.97      0.99      0.98      8454

    accuracy                           0.98     16481
   macro avg       0.98      0.98      0.98     16481
weighted avg       0.98      0.98      0.98     16481

